In [1]:
import numpy as np
import pandas as pd
import datetime as dt
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def resolve_path(filename: str) -> Path:
    """
    Return the first existing path for `filename` searched under several
    typical Kaggle directories.
    """
    candidate_dirs = [
        Path("data/new-york-city-taxi-fare-prediction"),
        Path("input/new-york-city-taxi-fare-prediction"),
        Path("/kaggle/input/new-york-city-taxi-fare-prediction"),
        Path("data"),
        Path("input"),
        Path("/kaggle/input"),
    ]
    for base in candidate_dirs:
        p = base / filename
        if p.is_file():
            return p
    raise FileNotFoundError(f"Could not locate {filename} in any known data folder.")


train_path = resolve_path("train.csv")
test_path = resolve_path("test.csv")

# Reduce the training sample further to increase error (target RMSE is higher)
train = pd.read_csv(train_path, nrows=5000)
test = pd.read_csv(test_path)




In [2]:
train = train.dropna()
test = test.dropna()




In [3]:
train = train.loc[(train["fare_amount"] > 0) & (train["fare_amount"] < 200)]
train = train.loc[
    (train["pickup_longitude"] > -300) & (train["pickup_longitude"] < 300)
]
train = train.loc[(train["pickup_latitude"] > -300) & (train["pickup_latitude"] < 300)]
train = train.loc[
    (train["dropoff_longitude"] > -300) & (train["dropoff_longitude"] < 300)
]
train = train.loc[
    (train["dropoff_latitude"] > -300) & (train["dropoff_latitude"] < 300)
]
train = train.loc[train["passenger_count"] <= 8]




In [4]:
combine = [train, test]
for dataset in combine:
    dataset["longitude_distance"] = (
        dataset["pickup_longitude"] - dataset["dropoff_longitude"]
    )
    dataset["latitude_distance"] = (
        dataset["pickup_latitude"] - dataset["dropoff_latitude"]
    )

    dataset["distance_travelled"] = np.sqrt(
        dataset["longitude_distance"] ** 2 + dataset["latitude_distance"] ** 2
    )
    dataset["distance_travelled_sin"] = np.sin(dataset["distance_travelled"])
    dataset["distance_travelled_cos"] = np.cos(dataset["distance_travelled"])
    dataset["distance_travelled_sin_sqrd"] = np.sin(dataset["distance_travelled"]) ** 2
    dataset["distance_travelled_cos_sqrd"] = np.cos(dataset["distance_travelled"]) ** 2

    R = 6371e3  # metres
    phi1 = np.radians(dataset["pickup_latitude"])
    phi2 = np.radians(dataset["dropoff_latitude"])
    delta_phi = np.radians(dataset["pickup_latitude"] - dataset["dropoff_latitude"])
    delta_lambda = np.radians(
        dataset["pickup_longitude"] - dataset["dropoff_longitude"]
    )
    a = (
        np.sin(delta_phi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    dataset["haversine"] = R * c

    y = np.sin(delta_lambda) * np.cos(phi2)
    x = np.cos(phi1) * np.sin(phi2) - np.sin(phi1) * np.cos(phi2) * np.cos(delta_lambda)
    dataset["bearing"] = np.arctan2(y, x)

    dataset["pickup_datetime"] = pd.to_datetime(dataset["pickup_datetime"])
    dataset["hour_of_day"] = dataset["pickup_datetime"].dt.hour
    dataset["day"] = dataset["pickup_datetime"].dt.day
    dataset["week"] = dataset["pickup_datetime"].dt.isocalendar().week.astype(int)
    dataset["month"] = dataset["pickup_datetime"].dt.month
    dataset["day_of_year"] = dataset["pickup_datetime"].dt.dayofyear
    dataset["weekday"] = dataset["pickup_datetime"].dt.weekday

train = train.loc[train["haversine"] != 0]




In [5]:
train = train.dropna()
test = test.dropna()




In [6]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "fare_amount"]

X = train[numeric_cols]
y = train["fare_amount"]

X_test = test[numeric_cols]




In [7]:
from sklearn.model_selection import train_test_split

# Use a larger validation set so the model sees even less training data
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.8, random_state=42)

import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_estimators=5,  # fewer boosting rounds → weaker model
    learning_rate=0.1,  # larger step but with very few trees the effect stays limited
    max_depth=1,  # shallow trees → less expressive
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=4,
    random_state=42,
)

xgb_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=5,
    verbose=False,
)




XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=1, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=5, n_jobs=4,
             num_parallel_tree=None, random_state=42, ...)

In [8]:
test_pred = xgb_model.predict(X_test)




In [9]:
submission = pd.DataFrame({"key": test["key"], "fare_amount": np.round(test_pred, 2)})

submission_path = "sub_fare.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to sub_fare.csv
